# JASP → MLIR demo

This notebook demonstrates the **JASP MLIR emission pipeline** from the `jasp_next` branch of Qrisp.

It shows how a Qrisp quantum program is:
1. Traced to a JAX expression (`jaspr`)
2. Lowered to MLIR with proper JASP dialect ops
3. Structured into `jasp.module` / `jasp.quantum_kernel` / `jasp.call` for QPU/CPU separation

No local install needed — everything runs here in Colab.

## 0. Install

In [ ]:
%pip uninstall torch sympy -y
%pip install sympy==1.13
%pip install xdsl
%pip install -q git+https://github.com/bruno-ah-um/Qrisp.git@jasp_next

## 1. Basic MLIR emission

A simple circuit: allocate 3 qubits, apply H + CX, measure. The resulting MLIR shows `jasp.create_qubits`, `jasp.quantum_gate`, and `jasp.measure` printed without quotes — confirming the JASP dialect is properly registered in xDSL.

In [1]:
from qrisp import QuantumVariable, h, cx, measure
from qrisp.jasp import make_jaspr

def main():
    qv = QuantumVariable(3)
    h(qv[0])
    cx(qv[0], qv[1])
    cx(qv[0], qv[2])
    return measure(qv)

jaspr = make_jaspr(main)()
xdsl_module = jaspr.to_mlir()
print(xdsl_module)

builtin.module @jasp_module {
  func.func @main() -> tensor<i64> {
    %0 = jasp.call @main_kernel() : () -> tensor<i64>
    func.return %0 : tensor<i64>
  }
  jasp.module @qpu_module{
    jasp.quantum_kernel @main_kernel(%arg14 : !jasp.QuantumState) -> tensor<i64> {
      %0 = "stablehlo.constant"() <{value = dense<3> : tensor<i64>}> : () -> tensor<i64>
      %1, %2 = jasp.create_qubits %0, %arg14 : !jasp.QuantumState, tensor<i64> -> !jasp.QubitArray, !jasp.QuantumState
      %3 = "stablehlo.constant"() <{value = dense<0> : tensor<i64>}> : () -> tensor<i64>
      %4 = jasp.get_qubit %1, %3 : !jasp.QubitArray, tensor<i64> -> !jasp.Qubit
      %5 = jasp.quantum_gate "h" (%4) , %2 : (!jasp.Qubit) , !jasp.QuantumState -> !jasp.QuantumState
      %6 = "stablehlo.constant"() <{value = dense<1> : tensor<i64>}> : () -> tensor<i64>
      %7 = jasp.get_qubit %1, %6 : !jasp.QubitArray, tensor<i64> -> !jasp.Qubit
      %8 = jasp.quantum_gate "cx" (%4, %7) , %5 : (!jasp.Qubit, !jasp.Qubit) , !jasp

## 2. Quantum control flow

JAX normally emits `stablehlo.case` / `stablehlo.while` for control flow. The `fix_quantum_control_flow` pass rewrites these to `scf.if` / `scf.while` when they carry `!jasp.QuantumState` — necessary because `stablehlo` control flow cannot hold quantum types.

In [2]:
from qrisp import QuantumVariable, QuantumFloat, h, x, rz, cx, measure, control
from qrisp.jasp import make_jaspr, jrange

def main():
    qv = QuantumVariable(3)
    h(qv[0])
    c = measure(qv[0])          # classical bit from mid-circuit measurement
    with control(c):             # measurement-controlled gate
        x(qv[1])
    for i in jrange(qv.size):   # dynamic loop over qubit count
        h(qv[i])

jaspr = make_jaspr(main)()
xdsl_module = jaspr.to_mlir()
mlir_str = str(xdsl_module)

print("stablehlo.case  present:", "stablehlo.case"  in mlir_str)  # should be False
print("stablehlo.while present:", "stablehlo.while" in mlir_str)  # should be False
print("scf.if present:         ", "scf.if"          in mlir_str)  # should be True
print()
print(mlir_str)

stablehlo.case  present: False
stablehlo.while present: False
scf.if present:          True

builtin.module @jasp_module {
  func.func @main() {
    %0, %1, %2, %3 = jasp.call @main_kernel() : () -> (tensor<i1>, tensor<i64>, tensor<i64>, !jasp.QubitArray)
    %4 = "stablehlo.convert"(%0) : (tensor<i1>) -> tensor<i32>
    %5 = tensor.extract %4[] : tensor<i32>
    %6 = arith.constant 0 : i32
    %7 = arith.cmpi ne, %5, %6 : i32
    %8 = scf.if %7 -> (!jasp.QuantumState) {
      scf.yield %9 : !jasp.QuantumState
    } else {
      %10 = "stablehlo.constant"() <{value = dense<1> : tensor<i64>}> : () -> tensor<i64>
      %11 = jasp.get_qubit %12, %10 : !jasp.QubitArray, tensor<i64> -> !jasp.Qubit
      %13 = jasp.quantum_gate "x" (%11) , %9 : (!jasp.Qubit) , !jasp.QuantumState -> !jasp.QuantumState
      scf.yield %13 : !jasp.QuantumState
    }
    %14 = "stablehlo.subtract"(%1, %2) : (tensor<i64>, tensor<i64>) -> tensor<i64>
    %15 = "stablehlo.subtract"(%14, %14) : (tensor<i64>, tensor<

## 3. `quantum_kernel`: QPU / CPU separation

The `@quantum_kernel` decorator marks a function as QPU code. The pipeline:

1. **`lift_quantum_kernels`** — promotes the function from `func.func` to `jasp.quantum_kernel` inside a `jasp.module`, and replaces the call site with a classical `jasp.call`.
2. **`hoist_classical_ops`** — moves non-QPU-safe ops (e.g. `stablehlo.convert`, `stablehlo.multiply`) out of the kernel body and into `@main`, where they run on the CPU host after the QPU returns.
3. **`drop_dead_wrappers`** — erases unused `private func.func` stubs emitted by JAX.

The result: the kernel body contains **only** `jasp.*` ops and `stablehlo.constant`; everything else is in `@main`.

In [3]:
from qrisp import QuantumFloat, measure
from qrisp.jasp import make_jaspr, quantum_kernel
from qrisp.jasp.mlir.mlir_emission import jaspr_to_mlir
import jax.numpy as jnp

@quantum_kernel
def sampling_kernel(k):
    """Runs on the QPU: allocate qubits, apply gates, measure."""
    qf = QuantumFloat(4)
    return measure(qf)

def main(k):
    return sampling_kernel(k)

jaspr = make_jaspr(main)(jnp.int64(1))
xdsl_module = jaspr_to_mlir(jaspr)
mlir_str = str(xdsl_module)

print("jasp.module present:         ", "jasp.module"          in mlir_str)
print("jasp.quantum_kernel present: ", "jasp.quantum_kernel"  in mlir_str)
print("jasp.call present:           ", "jasp.call"            in mlir_str)
print("jasp.return present:         ", "jasp.return"          in mlir_str)
print()
print(mlir_str)

jasp.module present:          True
jasp.quantum_kernel present:  True
jasp.call present:            True
jasp.return present:          True

builtin.module @jasp_module {
  func.func public @main(%arg11 : tensor<i64>, %arg12 : !jasp.QuantumState) -> (tensor<f64>, !jasp.QuantumState) {
    %0, %1 = jasp.call @sampling_kernel(%arg11) : (tensor<i64>) -> (tensor<i64>, tensor<f64>)
    %2 = "stablehlo.convert"(%0) : (tensor<i64>) -> tensor<f64>
    %3 = "stablehlo.multiply"(%2, %1) : (tensor<f64>, tensor<f64>) -> tensor<f64>
    func.return %3, %arg12 : tensor<f64>, !jasp.QuantumState
  }
  jasp.module @qpu_module{
    jasp.quantum_kernel private @sampling_kernel(%arg9 : tensor<i64>, %arg10 : !jasp.QuantumState) -> (tensor<i64>, tensor<f64>) {
      %0 = "stablehlo.constant"() <{value = dense<4> : tensor<i64>}> : () -> tensor<i64>
      %1, %2 = jasp.create_qubits %0, %arg10 : !jasp.QuantumState, tensor<i64> -> !jasp.QubitArray, !jasp.QuantumState
      %3, %4 = jasp.measure %1, %2 : !jasp.